# Baseline classifiers

This notebook evaluates two baselines for arrival-delay classification. Separate classifiers are fitted for each train type (IC, ICE, and RE) using the training partition and evaluated on the held-out test partition.

The majority baseline always predicts the most common target class. The class-share baseline samples predictions according to the class proportions observed in training. Macro-F1 is the primary metric because it gives each target class equal weight.

In [30]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.metrics import confusion_matrix
PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from baselines import MajorityClassClassifier, RandomClassShareClassifier
from data_utils import (
    TARGET_LABELS,
    add_delay_target,
    chronological_split,
    load_monthly_data,
)
from model_evaluation import macro_f1_test

## Load the labelled data

Only the columns needed to construct the target and split the data are loaded. The source month is taken from the file name so that the split is chronological.

In [31]:
DATA_DIR = PROJECT_ROOT / "data"
BASELINE_COLUMNS = [
    "delay_in_min",
    "arrival_is_canceled",
    "arrival_planned_time",
    "train_type",
]

data = load_monthly_data(DATA_DIR, BASELINE_COLUMNS)
labelled = add_delay_target(data)

print(f"Labelled rows: {len(labelled):,}")
labelled["source_file"].value_counts().sort_index().to_frame("rows")

Labelled rows: 13,292,608


,rows
source_file,
data-2026-01.parquet,1591674
data-2026-02.parquet,1399550
data-2026-03.parquet,1483044
data-2026-04.parquet,1430929
data-2026-05.parquet,1472474
data-2026-06.parquet,1477827
data-2026-07.parquet,1446007
data-2026-08.parquet,1485514
data-2026-09.parquet,1505589


## Chronological train/validation/test split

The data is split chronologically into train (70%), validation (15%), and test (15%) partitions. The baseline is fitted only on the training partition; validation and test rows are not used for fitting.

In [32]:
train, validation, test = chronological_split(labelled)

print(f"Training rows: {len(train):,}")
print(f"Validation rows: {len(validation):,}")
print(f"Test rows: {len(test):,}")
print("\nTraining target distribution by train type:")
display(pd.crosstab(train["train_type"], train["target"], normalize="index").round(4))

Training rows: 9,304,825
Validation rows: 1,993,891
Test rows: 1,993,892

Training target distribution by train type:


target,delayed,late,on_time,severe_or_cancelled,slight_delay
train_type,,,,,
IC,0.1749,0.2296,0.2744,0.1893,0.1318
ICE,0.1928,0.2034,0.2920,0.1916,0.1201
RE,0.0983,0.2086,0.4640,0.0560,0.1732


## Baseline classifiers by train type

Each train type receives separate majority and class-share baselines. A constant feature is used because neither baseline learns from input features.

In [33]:
results = []
confusion_matrices = {}

for train_type in ["IC", "ICE", "RE"]:
    train_group = train[train["train_type"] == train_type]
    test_group = test[test["train_type"] == train_type]

    if train_group.empty or test_group.empty:
        raise ValueError(f"Missing training or test rows for train type: {train_type}")

    constant_train = np.zeros((len(train_group), 1))
    constant_test = np.zeros((len(test_group), 1))
    models = {
        "majority": MajorityClassClassifier(),
        "class_share": RandomClassShareClassifier(random_state=42),
    }

    for baseline_name, model in models.items():
        model.fit(constant_train, train_group["target"])
        predictions = model.predict(constant_test)
        confusion_matrices[(train_type, baseline_name)] = confusion_matrix(
            test_group["target"], predictions, labels=TARGET_LABELS
        )
        result = macro_f1_test(
            model,
            constant_test,
            test_group["target"],
            model_name=f"{baseline_name}_{train_type}",
        )
        result["train_type"] = train_type
        result["baseline"] = baseline_name
        result["training_class_shares"] = (
            train_group["target"].value_counts(normalize=True).round(4).to_dict()
        )
        results.append(result)

baseline_results = (
    pd.DataFrame(results)
    [["train_type", "baseline", "model", "training_class_shares", "n_test", "macro_f1", "accuracy"]]
    .sort_values(["train_type", "baseline"])
    .reset_index(drop=True)
)
baseline_results

,train_type,baseline,model,training_class_shares,n_test,macro_f1,accuracy
0,IC,class_share,class_share_IC,"{'on_time': 0.2744, 'late': 0.2296, 'severe_or...",68419,0.196553,0.208860
1,IC,majority,majority_IC,"{'on_time': 0.2744, 'late': 0.2296, 'severe_or...",68419,0.084024,0.265920
2,ICE,class_share,class_share_ICE,"{'on_time': 0.292, 'late': 0.2034, 'delayed': ...",236620,0.199522,0.211098
3,ICE,majority,majority_ICE,"{'on_time': 0.292, 'late': 0.2034, 'delayed': ...",236620,0.077949,0.242038
4,RE,class_share,class_share_RE,"{'on_time': 0.464, 'late': 0.2086, 'slight_del...",1688853,0.199883,0.298882
5,RE,majority,majority_RE,"{'on_time': 0.464, 'late': 0.2086, 'slight_del...",1688853,0.124333,0.451029


## Confusion matrices

Rows show the true class and columns show the predicted class. These matrices reveal which classes each baseline confuses, including the effect of always predicting one class or making random class-share predictions.

In [34]:
for train_type in ["IC", "ICE", "RE"]:
    for baseline_name in ["majority", "class_share"]:
        print(f"{train_type} - {baseline_name}")
        display(
            pd.DataFrame(
                confusion_matrices[(train_type, baseline_name)],
                index=pd.Index(TARGET_LABELS, name="true"),
                columns=pd.Index(TARGET_LABELS, name="predicted"),
            )
        )


IC - majority


predicted,on_time,slight_delay,late,delayed,severe_or_cancelled
true,,,,,
on_time,18194,0,0,0,0
slight_delay,7562,0,0,0,0
late,15031,0,0,0,0
delayed,13331,0,0,0,0
severe_or_cancelled,14301,0,0,0,0


IC - class_share


predicted,on_time,slight_delay,late,delayed,severe_or_cancelled
true,,,,,
on_time,5045,2339,4205,3144,3461
slight_delay,2098,979,1752,1322,1411
late,4007,2086,3419,2615,2904
delayed,3690,1741,2959,2357,2584
severe_or_cancelled,3964,1832,3214,2544,2747


ICE - majority


...

predicted,on_time,slight_delay,late,delayed,severe_or_cancelled
true,,,,,
on_time,16743,6789,11829,10925,10985
slight_delay,7711,3215,5421,5049,5078
late,14583,6136,10186,9835,9773
delayed,16397,6657,11112,10575,10486
severe_or_cancelled,13687,5683,9565,9106,9094


RE - majority


predicted,on_time,slight_delay,late,delayed,severe_or_cancelled
true,,,,,
on_time,761721,0,0,0,0
slight_delay,306052,0,0,0,0
late,374295,0,0,0,0
delayed,164702,0,0,0,0
severe_or_cancelled,82083,0,0,0,0


RE - class_share


predicted,on_time,slight_delay,late,delayed,severe_or_cancelled
true,,,,,
on_time,353810,132097,158658,74456,42700
slight_delay,141936,53160,63940,29873,17143
late,173222,64787,78180,37111,20995
delayed,76543,28277,34432,16212,9238
severe_or_cancelled,38166,14052,17401,7857,4607


## Summary

The baseline results show why macro-F1 is the main metric. The majority baseline reaches 45.10% accuracy for RE, but its macro-F1 is only 0.124 because it predicts one class and gives the other classes an F1 score of zero. The same pattern appears for IC (26.59% accuracy, macro-F1 0.084) and ICE (24.20%, 0.078). The class-share baseline has lower accuracy in this run (20.89% for IC, 21.11% for ICE, and 29.89% for RE), but higher macro-F1 values (0.197, 0.200, and 0.200) because it predicts all classes and achieves some recall for minority classes. The confusion matrices make this visible: majority predictions are concentrated in one column, while class-share predictions are distributed across columns. Macro-F1 is therefore better than accuracy for this imbalanced multi-class task because it rewards performance across every delay class rather than mainly the most common class.